# Design Requirements: AI-Powered Customer Call Analysis in Snowflake

**Use Case:** Analyze transcribed customer service calls to identify specific situations, flag alerts, and escalate issues based on keyword identifiers and contextual analysis.

**Business Scenario:** Customer service setting where phone agents field inbound service calls. The system needs to automatically identify when specific topics (e.g., "baseball programs") are mentioned, exclude irrelevant contexts (e.g., "soccer"), and analyze word proximity to determine relevance and urgency.

**Objective:** Showcase the comprehensive use of Snowflake's AI/ML SQL functions (Cortex) to build an end-to-end intelligent analysis workflow for call center data, including real-time alerting and escalation mechanisms.

## Key Requirements Summary:
1. **Audio Transcription** - Convert call recordings to searchable text
2. **Content Classification** - Categorize calls by topic and urgency
3. **Contextual Filtering** - Identify relevant mentions with exclusion logic
4. **Proximity Analysis** - Detect keyword relationships within specified word distances
5. **Sentiment Analysis** - Assess customer emotion and satisfaction
6. **Alert Generation** - Create actionable escalations for management
7. **Automated Workflows** - Process calls in real-time as they arrive

## 1. Prerequisites and Setup

Before running the analysis, ensure the following setup is complete in your Snowflake environment.

**Requirements:**
1.  **Internal Named Stage for Audio Files:** A stage is needed to store the raw audio files (e.g., `.wav`, `.mp3`, `.flac`) of the customer calls.
2.  **Target Tables:** Multiple tables are required for different aspects of the analysis:
    - `call_transcriptions` - Raw transcribed text
    - `call_analysis` - AI-driven insights and classifications
    - `call_alerts` - Flagged calls requiring attention
    - `escalation_queue` - High-priority items for management review
3.  **Permissions:** The user's role must have the necessary privileges to create stages, tables, views, tasks, streams, and execute all Snowflake AI_SQL functions.
4.  **AI_SQL Functions Access:** Ensure access to all required AI functions:
    - `AI_SENTIMENT()` - Analyze emotional tone of text
    - `AI_CLASSIFY()` - Categorize content into predefined classes
    - `AI_COMPLETE()` - Generate text completions and perform analysis
    - Audio transcription will use pre-populated sample data

In [ ]:
-- Code to create the stage and tables for comprehensive call analysis
CREATE OR REPLACE STAGE call_center_audio;

-- Example: Upload sample audio files to the stage
-- PUT file:///path/to/sample_call.wav @call_center_audio;
-- PUT file:///path/to/baseball_inquiry.wav @call_center_audio;
-- PUT file:///path/to/soccer_discussion.wav @call_center_audio;

-- Main transcription table (compatible with AI_SQL functions)
CREATE OR REPLACE TABLE call_transcriptions (
    call_id STRING PRIMARY KEY,
    audio_file_name STRING NOT NULL,
    transcription TEXT,
    call_duration_seconds INTEGER,
    agent_id STRING,
    customer_phone STRING,
    customer_id STRING,
    call_start_time TIMESTAMP_LTZ,
    call_end_time TIMESTAMP_LTZ,
    created_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- AI analysis results table (compatible format)
CREATE OR REPLACE TABLE call_analysis (
    analysis_id STRING PRIMARY KEY,
    call_id STRING NOT NULL,
    call_category STRING,
    sentiment_score FLOAT,
    sentiment_label STRING,
    urgency_level STRING,
    key_topics VARCHAR, -- Comma-separated values instead of ARRAY
    customer_satisfaction_score FLOAT,
    requires_escalation BOOLEAN DEFAULT FALSE,
    confidence_score FLOAT,
    analysis_timestamp TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Alert and escalation tracking (simplified)
CREATE OR REPLACE TABLE call_alerts (
    alert_id STRING PRIMARY KEY,
    call_id STRING NOT NULL,
    alert_type STRING,
    alert_reason TEXT,
    priority_level STRING,
    assigned_to STRING,
    status STRING DEFAULT 'OPEN',
    resolution_notes TEXT,
    created_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(),
    resolved_at TIMESTAMP_LTZ
);

## 2. Sample Data Setup and Transcription

**Requirement:** Set up sample call data with pre-transcribed content for AI analysis.

**Approach:** Use realistic sample data with transcribed calls

**Design:**
- Populate the `call_transcriptions` table with sample call data
- Include baseball and soccer scenarios for testing exclusion logic
- Ensure data is ready for AI_SQL function analysis
- This represents the output that would come from audio transcription in production

In [ ]:
-- Insert expanded sample call transcriptions with diverse scenarios
INSERT INTO call_transcriptions (
    call_id, audio_file_name, transcription, call_duration_seconds, 
    agent_id, customer_phone, customer_id, call_start_time, call_end_time
) VALUES
    -- Baseball program inquiry (POSITIVE MATCH)
    ('CALL_001', 'baseball_inquiry_001.wav', 
     'Hi, I''m calling about the baseball program that was mentioned in your recent newsletter. My son is really interested in joining the little league program this summer. Can you tell me more about the schedule and registration process? I want to make sure we don''t miss the deadline for the program enrollment.',
     420, 'AGENT_002', '+1234567890', 'CUST_001', 
     '2024-03-15 09:15:00'::timestamp_ltz, '2024-03-15 09:22:00'::timestamp_ltz),
     
    -- Baseball equipment complaint (POSITIVE MATCH)  
    ('CALL_002', 'baseball_equipment_002.wav',
     'I''m really frustrated with the baseball equipment delivery. I ordered bats and gloves for our team''s program three weeks ago and they still haven''t arrived. The season program starts next week and we need this equipment urgently. This is affecting our entire training schedule.',
     315, 'AGENT_001', '+1234567891', 'CUST_002',
     '2024-03-15 14:30:00'::timestamp_ltz, '2024-03-15 14:35:15'::timestamp_ltz),
     
    -- Soccer discussion (NEGATIVE MATCH - should be excluded)
    ('CALL_003', 'soccer_discussion_003.wav',
     'Hello, I wanted to ask about the sports channel package. My daughter loves soccer and we want to watch the World Cup games. Do you have any soccer-specific packages? Also, I heard there might be some baseball games included, but we''re primarily interested in soccer coverage.',
     280, 'AGENT_003', '+1234567892', 'CUST_003',
     '2024-03-15 11:45:00'::timestamp_ltz, '2024-03-15 11:49:40'::timestamp_ltz),
     
    -- Baseball coaching program (POSITIVE MATCH)
    ('CALL_004', 'baseball_coaching_004.wav',
     'I''m interested in the baseball coaching certification program you offer. I''ve been coaching youth baseball for five years and want to advance my skills. Can you provide details about the program curriculum and the schedule for the upcoming sessions?',
     380, 'AGENT_002', '+1234567893', 'CUST_004',
     '2024-03-16 10:20:00'::timestamp_ltz, '2024-03-16 10:26:20'::timestamp_ltz),

    -- Baseball field rental (POSITIVE MATCH)
    ('CALL_005', 'baseball_field_005.wav',
     'Good morning, I''m calling to inquire about renting baseball fields for our company tournament. We need access to the baseball diamond for three weekends in April. What''s the rental process and are there any specific requirements for field usage?',
     245, 'AGENT_001', '+1234567894', 'CUST_005',
     '2024-03-16 08:30:00'::timestamp_ltz, '2024-03-16 08:34:05'::timestamp_ltz),

    -- Baseball camp registration urgent (POSITIVE MATCH)
    ('CALL_006', 'baseball_camp_006.wav',
     'Hi, I''m calling about the summer baseball camp program. My daughter really wants to attend but I just heard about it today. Is it too late to register? The program starts next month and I''m worried we missed the deadline. This is really important to her.',
     410, 'AGENT_003', '+1234567895', 'CUST_006',
     '2024-03-16 13:20:00'::timestamp_ltz, '2024-03-16 13:26:50'::timestamp_ltz),

    -- Baseball ticket complaint (POSITIVE MATCH)
    ('CALL_007', 'baseball_tickets_007.wav',
     'I''m extremely upset about the baseball season tickets I purchased. The seats we were promised are not what we received, and customer service has been unresponsive. I''ve been a loyal fan for 20 years and this treatment is unacceptable. I want this resolved immediately.',
     520, 'AGENT_004', '+1234567896', 'CUST_007',
     '2024-03-16 16:45:00'::timestamp_ltz, '2024-03-16 16:54:20'::timestamp_ltz),

    -- General sports inquiry mentioning baseball (POSITIVE MATCH)
    ('CALL_008', 'sports_general_008.wav',
     'I''m looking for information about your sports programs. My kids are interested in baseball and basketball. Can you tell me what programs are available for ages 8-12? I''m particularly interested in the baseball options since that''s their favorite sport.',
     290, 'AGENT_002', '+1234567897', 'CUST_008',
     '2024-03-17 10:15:00'::timestamp_ltz, '2024-03-17 10:19:50'::timestamp_ltz),

    -- Baseball merchandise return (POSITIVE MATCH)
    ('CALL_009', 'baseball_merch_009.wav',
     'I need to return some baseball merchandise I ordered online. The jerseys don''t fit properly and one of the baseball caps has a defect. What''s your return policy for baseball equipment and apparel? I have the order number ready.',
     195, 'AGENT_001', '+1234567898', 'CUST_009',
     '2024-03-17 14:10:00'::timestamp_ltz, '2024-03-17 14:13:15'::timestamp_ltz),

    -- Mixed sports with soccer priority (NEGATIVE MATCH)
    ('CALL_010', 'mixed_sports_010.wav',
     'Hi, I''m interested in your youth sports programs. My son loves soccer and that''s his main sport, but he also enjoys baseball occasionally. The soccer program is our priority though. Do you have competitive soccer teams? We might consider baseball as a secondary option.',
     330, 'AGENT_003', '+1234567899', 'CUST_010',
     '2024-03-17 11:30:00'::timestamp_ltz, '2024-03-17 11:35:30'::timestamp_ltz),

    -- Baseball volunteer opportunity (POSITIVE MATCH)
    ('CALL_011', 'baseball_volunteer_011.wav',
     'I''d like to volunteer for the local baseball league. I have experience playing college baseball and would love to help with coaching or field maintenance. What volunteer opportunities are available for the baseball program this season?',
     275, 'AGENT_004', '+1234567800', 'CUST_011',
     '2024-03-18 09:45:00'::timestamp_ltz, '2024-03-18 09:49:35'::timestamp_ltz),

    -- Baseball injury/medical (POSITIVE MATCH)
    ('CALL_012', 'baseball_injury_012.wav',
     'My son was injured during yesterday''s baseball practice and I need information about the insurance coverage for baseball program participants. The injury happened during an official practice session. What''s the process for filing a claim?',
     395, 'AGENT_001', '+1234567801', 'CUST_012',
     '2024-03-18 15:20:00'::timestamp_ltz, '2024-03-18 15:26:35'::timestamp_ltz),

    -- Non-sports billing call (NEGATIVE MATCH)
    ('CALL_013', 'billing_general_013.wav',
     'I''m calling about my monthly subscription billing. There seems to be an extra charge on my account that I don''t understand. Can you help me review my recent billing statements and explain these charges? This is not related to any sports programs.',
     240, 'AGENT_002', '+1234567802', 'CUST_013',
     '2024-03-18 12:10:00'::timestamp_ltz, '2024-03-18 12:14:00'::timestamp_ltz),

    -- Baseball tournament registration (POSITIVE MATCH)
    ('CALL_014', 'baseball_tournament_014.wav',
     'We''re organizing a regional baseball tournament and want to register our team. This is for adult recreational baseball league. What are the tournament fees and registration requirements? The tournament is scheduled for late summer.',
     355, 'AGENT_003', '+1234567803', 'CUST_014',
     '2024-03-19 08:15:00'::timestamp_ltz, '2024-03-19 08:21:10'::timestamp_ltz),

    -- Technical support unrelated (NEGATIVE MATCH)
    ('CALL_015', 'tech_support_015.wav',
     'I''m having trouble with my internet connection and need technical support. The wifi keeps dropping and it''s affecting my work from home setup. Can you send a technician to check the connection? This has been going on for three days.',
     310, 'AGENT_004', '+1234567804', 'CUST_015',
     '2024-03-19 14:30:00'::timestamp_ltz, '2024-03-19 14:35:10'::timestamp_ltz);

-- Verify the sample data is loaded correctly
SELECT 
    call_id,
    audio_file_name,
    LENGTH(transcription) as transcript_length,
    LEFT(transcription, 100) as transcript_preview,
    agent_id,
    created_at
FROM call_transcriptions
ORDER BY created_at DESC;

## 3. Call Classification with `AI_CLASSIFY`

**Requirement:** Perform an initial categorization of calls to quickly identify common topics. This helps in routing and high-level analytics.

**Function:** `AI_CLASSIFY()`

**Design:**
- Define a set of relevant labels for call classification (e.g., 'Sales Inquiry', 'Technical Support', 'Billing Question', 'Complaint').
- Use the `AI_CLASSIFY` function with ARRAY_CONSTRUCT syntax to assign the most appropriate label to each transcription.
- Use LEFT() truncation to prevent VARCHAR length errors.
- The results can be stored in a new table or used for ad-hoc analysis.

In [ ]:
-- Classify the call transcriptions using AI_CLASSIFY with proper syntax
SELECT 
    call_id,
    LEFT(transcription, 100) as transcript_preview,
    AI_CLASSIFY(
        LEFT(transcription, 4000),  -- Truncate to prevent VARCHAR errors
        ARRAY_CONSTRUCT('Sales Inquiry', 'Technical Support', 'Billing Question', 'Sports Program Inquiry', 'Equipment Complaint', 'General Conversation')
    ) AS call_category
FROM 
    call_transcriptions;

-- Multi-label classification for call topics including baseball detection
SELECT 
    call_id,
    AI_CLASSIFY(
        LEFT(transcription, 4000),
        ARRAY_CONSTRUCT('Baseball', 'Soccer', 'Equipment', 'Billing', 'Technical', 'Program Registration')
    ) as primary_topic,
    AI_CLASSIFY(
        LEFT(transcription, 4000),
        ARRAY_CONSTRUCT('Urgent', 'Normal', 'Low Priority')
    ) as urgency_classification
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball') OR CONTAINS(LOWER(transcription), 'soccer');

## 4. Advanced Analysis and Filtering

**Requirement:** Identify calls that meet specific, nuanced criteria that go beyond simple keyword matching. The primary use case is to flag calls where "baseball" is mentioned, but not "soccer," and where "baseball" is mentioned within a specific word proximity to other key terms.

**Functions:** `AI_COMPLETE()` for all AI analysis, and standard SQL functions.

**Design:**
1.  **Exclusionary Filtering:** Use standard `WHERE` clauses with `CONTAINS()` to include transcripts mentioning "baseball" while excluding those mentioning "soccer".
2.  **Proximity Analysis:** Use the `AI_COMPLETE` function with a model name and carefully crafted prompt to determine if "baseball" is mentioned within a specified number of words (e.g., 50 words) of other keywords like "program" or "schedule". The model's response ('Yes' or 'No') can then be used for filtering.
3.  **Sentiment Analysis:** Use `AI_COMPLETE()` with targeted prompts to assess customer emotion and generate both sentiment score (-1.0 to +1.0) and label (positive/negative/neutral).
4.  **Text Truncation:** Always use LEFT() function to limit text length and prevent VARCHAR errors.

In [ ]:
-- First, let's check what data we have and verify the filtering works
SELECT 
    call_id,
    LEFT(transcription, 100) as transcript_preview,
    CASE 
        WHEN CONTAINS(LOWER(transcription), 'baseball') THEN 'HAS_BASEBALL'
        ELSE 'NO_BASEBALL'
    END as has_baseball,
    CASE 
        WHEN CONTAINS(LOWER(transcription), 'soccer') THEN 'HAS_SOCCER'
        ELSE 'NO_SOCCER'
    END as has_soccer
FROM call_transcriptions
ORDER BY created_at DESC;

-- Create a simplified view first to test the basic filtering
CREATE OR REPLACE VIEW baseball_calls_basic AS
SELECT
    call_id,
    customer_phone,
    agent_id,
    transcription,
    created_at,
    
    -- Check for exclusion criteria (soccer mentions)
    CASE 
        WHEN CONTAINS(LOWER(transcription), 'soccer') THEN 'EXCLUDED'
        ELSE 'INCLUDED'
    END as filter_status
    
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball')
ORDER BY created_at DESC;

-- Test the basic view first
SELECT call_id, customer_phone, filter_status, LEFT(transcription, 100) as preview
FROM baseball_calls_basic;

-- Now create the full view with AI functions (only if basic view works)
CREATE OR REPLACE VIEW baseball_alerts AS
SELECT
    call_id,
    customer_phone,
    agent_id,
    
    -- Use AI_COMPLETE for proximity analysis (with proper model parameter)
    AI_COMPLETE(
        'llama2-70b-chat',  -- Model name required as first parameter
        CONCAT(
            'Analyze this customer service call and determine if the word "baseball" ',
            'appears within 50 words of either "program" or "schedule". ',
            'Respond with only "Yes" or "No". ',
            'Transcript: ', LEFT(transcription, 4000)  -- Truncate to prevent errors
        )
    ) AS is_baseball_near_program,
    
    -- Extract sentiment information using AI_COMPLETE (more reliable than AI_SENTIMENT)
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Rate this customer call sentiment from -1.0 (very negative) to +1.0 (very positive). Respond with only a decimal number: ', LEFT(transcription, 2000))
    )::STRING as sentiment_score,
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('What is the sentiment of this call? Respond with only one word: positive, negative, or neutral. Text: ', LEFT(transcription, 2000))
    )::STRING as sentiment_label,
    
    -- Generate summary using AI_COMPLETE (since AI_SUMMARIZE may not be available)
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Summarize this baseball-related call in 1-2 sentences: ', LEFT(transcription, 4000))
    ) as summary,
    
    -- Check for exclusion criteria (soccer mentions)
    CASE 
        WHEN CONTAINS(LOWER(transcription), 'soccer') THEN 'EXCLUDED'
        ELSE 'INCLUDED'
    END as filter_status,
    
    created_at
FROM
    call_transcriptions
WHERE
    CONTAINS(LOWER(transcription), 'baseball')
ORDER BY created_at DESC;

-- Query ALL baseball calls first (without the proximity filter)
SELECT 
    call_id,
    customer_phone,
    filter_status,
    LEFT(summary, 100) as summary_preview
FROM 
    baseball_alerts
WHERE filter_status = 'INCLUDED';

-- Then query with the proximity filter (this might be empty if AI returns 'No')
SELECT 
    call_id,
    customer_phone,
    sentiment_score,
    sentiment_label,
    is_baseball_near_program,
    LEFT(summary, 100) as summary_preview
FROM 
    baseball_alerts
WHERE
    filter_status = 'INCLUDED' 
    AND TRIM(UPPER(is_baseball_near_program)) = 'YES';

## 5. Information Extraction and Analysis

**Requirement:** Extract specific information from calls and generate summaries for agents or managers to review quickly.

**Functions:** `AI_COMPLETE()` for both extraction and summarization

**Design:**
- Use `AI_COMPLETE()` with targeted prompts to extract specific information from transcriptions.
- Generate summaries using `AI_COMPLETE()` with summarization prompts.
- Extract customer requests, timeline information, and program details.
- Always use LEFT() truncation to prevent VARCHAR length errors.

In [ ]:
-- First, test AI_SENTIMENT to see the raw output structure
SELECT 
    call_id,
    LEFT(transcription, 100) as transcript_preview,
    AI_SENTIMENT(LEFT(transcription, 1000)) as raw_sentiment_output
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball')
LIMIT 1;

-- Extract specific information from call transcripts using AI_COMPLETE
SELECT 
    call_id,
    LEFT(transcription, 100) as transcript_preview,
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('What program is mentioned? Answer briefly: ', LEFT(transcription, 1000))
    ) as program_inquiry,
    AI_COMPLETE(
        'llama2-70b-chat', 
        CONCAT('Main issue? Answer briefly: ', LEFT(transcription, 1000))
    ) as main_concern,
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Any timeline mentioned? Yes/No: ', LEFT(transcription, 1000))
    ) as timeline_mentioned
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball');

-- Generate comprehensive summaries with reliable sentiment analysis using AI_COMPLETE
SELECT 
    call_id,
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Summarize this customer service call in 2-3 sentences: ', LEFT(transcription, 4000))
    ) as detailed_summary,
    
    -- Use AI_COMPLETE for reliable sentiment analysis
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Analyze the sentiment of this customer service call. Respond with only one word: positive, negative, or neutral. Text: ', LEFT(transcription, 2000))
    ) as sentiment_label,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Rate the customer sentiment on a scale from -1.0 (very negative) to +1.0 (very positive). Respond with only a decimal number: ', LEFT(transcription, 2000))
    ) as sentiment_score,
    
    -- Extract urgency level
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('What is the urgency level of this call? Respond with only: low, medium, or high. Text: ', LEFT(transcription, 1500))
    ) as urgency_level,
    
    -- Determine if escalation is needed
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Does this call require escalation to management? Respond with only: yes or no. Text: ', LEFT(transcription, 1500))
    ) as requires_escalation
    
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball')
  AND NOT CONTAINS(LOWER(transcription), 'soccer');

-- Create a simplified analysis view for easy querying
CREATE OR REPLACE VIEW call_analysis_results AS
SELECT 
    call_id,
    customer_phone,
    agent_id,
    created_at,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Summarize this call briefly: ', LEFT(transcription, 3000))
    ) as summary,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Sentiment (positive/negative/neutral): ', LEFT(transcription, 2000))
    ) as sentiment,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Urgency (low/medium/high): ', LEFT(transcription, 1500))
    ) as urgency,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('What program or service is mentioned? Answer briefly: ', LEFT(transcription, 1000))
    ) as program_mentioned
    
FROM call_transcriptions
WHERE transcription IS NOT NULL;

-- Query the simplified view for baseball-related calls
SELECT * FROM call_analysis_results 
WHERE CONTAINS(LOWER(program_mentioned), 'baseball') 
   OR call_id IN (
       SELECT call_id FROM call_transcriptions 
       WHERE CONTAINS(LOWER(transcription), 'baseball')
         AND NOT CONTAINS(LOWER(transcription), 'soccer')
   )
ORDER BY created_at DESC;

## 6. Advanced Search and Pattern Matching

**Requirement:** Enable search capabilities over the analyzed call data to find similar patterns and historical cases.

**Approach:** SQL-based keyword search with relevance scoring (reliable alternative to Cortex Search)

**Design:**
- Use SQL CONTAINS() functions for keyword matching
- Implement relevance scoring based on keyword combinations
- Support complex search patterns for baseball program inquiries and related cases
- Provide ranked results based on content relevance


In [ ]:
-- Step 1: Create optimized enriched view with efficient AI usage (Performance Optimized)
CREATE OR REPLACE VIEW call_search_data AS
WITH ai_analysis AS (
    SELECT 
        call_id,
        customer_phone,
        agent_id,
        created_at,
        transcription,
        
        -- Efficient AI analysis with shorter text inputs for speed
        AI_CLASSIFY(
            LEFT(transcription, 2000),  -- Reduced from 4000 for faster processing
            ARRAY_CONSTRUCT('Baseball Program', 'Equipment Issue', 'Billing Question', 'Technical Support', 'General Inquiry')
        )::STRING as call_category,
        
        AI_COMPLETE(
            'llama2-70b-chat',
            CONCAT('Sentiment: ', LEFT(transcription, 1000))  -- Shortened for speed
        )::STRING as sentiment_label,
        
        AI_COMPLETE(
            'llama2-70b-chat',
            CONCAT('Summary: ', LEFT(transcription, 2000))  -- Reduced from 4000
        )::STRING as ai_summary
        
    FROM call_transcriptions
    WHERE transcription IS NOT NULL
      AND CONTAINS(LOWER(transcription), 'baseball')  -- Filter early for performance
      AND NOT CONTAINS(LOWER(transcription), 'soccer')
)
SELECT 
    call_id,
    customer_phone,
    agent_id,
    created_at,
    transcription,
    call_category,
    sentiment_label,
    ai_summary,
    
    -- Optimized searchable content using already computed values (no duplicate AI calls)
    CONCAT(
        LEFT(transcription, 1000), ' ',  -- Shorter transcription for performance
        'Category: ', call_category, ' ',
        'Summary: ', ai_summary, ' ',
        'Sentiment: ', sentiment_label
    ) as enriched_content
    
FROM ai_analysis;

-- Step 2: Create optimized materialized table with actual AI analysis (Performance Focused)
CREATE OR REPLACE TABLE call_search_materialized AS
SELECT 
    call_id,
    customer_phone, 
    agent_id,
    created_at,
    LEFT(transcription, 2000) as transcription_excerpt,  -- Store shorter version for search
    
    -- Real AI analysis with performance optimizations
    AI_CLASSIFY(
        LEFT(transcription, 1500),  -- Even shorter for materialization speed
        ARRAY_CONSTRUCT('Program Inquiry', 'Equipment Issue', 'Complaint', 'General')
    )::STRING as call_category,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Sentiment (positive/negative/neutral): ', LEFT(transcription, 800))
    )::STRING as sentiment_label,
    
    AI_COMPLETE(
        'llama2-70b-chat',
        CONCAT('Brief summary: ', LEFT(transcription, 1500))
    )::STRING as ai_summary,
    
    -- Optimized searchable content
    CONCAT(
        LEFT(transcription, 1500), ' ',
        'Type: baseball program inquiry'
    ) as enriched_content
    
FROM call_transcriptions
WHERE CONTAINS(LOWER(transcription), 'baseball')
  AND NOT CONTAINS(LOWER(transcription), 'soccer')
  AND LENGTH(transcription) > 50;  -- Skip very short transcriptions

-- Step 3: Create Cortex Search Service over the materialized table (using Default_WH)
CREATE OR REPLACE CORTEX SEARCH SERVICE call_center_search
ON enriched_content
ATTRIBUTES call_id, customer_phone, agent_id, call_category, sentiment_label, ai_summary, created_at
WAREHOUSE = Default_WH
TARGET_LAG = '1 minute'
AS (
    SELECT 
        call_id,
        customer_phone, 
        agent_id,
        call_category,
        sentiment_label,
        ai_summary,
        created_at,
        enriched_content
    FROM call_search_materialized
);

-- Alternative: Simple search over transcriptions only (most reliable)
CREATE OR REPLACE CORTEX SEARCH SERVICE call_transcription_search
ON transcription
ATTRIBUTES call_id, customer_phone, agent_id, created_at
WAREHOUSE = Default_WH
TARGET_LAG = '1 minute'
AS (
    SELECT 
        call_id,
        customer_phone,
        agent_id,
        created_at,
        transcription
    FROM call_transcriptions
    WHERE transcription IS NOT NULL
);


## 7. Conclusion and Next Steps

This notebook design outlines a complete workflow for leveraging Snowflake AI_SQL functions to derive actionable insights from call center data. 

**Summary of Requirements Met:**
- High-level classification of call topics using `AI_CLASSIFY()`.
- Specific, rule-based filtering (e.g., mentions 'baseball' but not 'soccer').
- Advanced proximity analysis using `AI_COMPLETE()` with proper model parameters.
- Sentiment analysis with `AI_SENTIMENT()` and object field extraction.
- Information extraction and summarization using `AI_COMPLETE()`.
- **Cortex Search integration** for semantic search over AI-enriched call data.


**Potential Next Steps:**
1.  **Automation:** Wrap the SQL logic in Snowflake Tasks and Streams to automatically process new calls as they arrive.
2.  **Alerting:** Use Snowflake Alerts to send notifications when new records appear in the `baseball_alerts` view.
3.  **Visualization:** Build a Snowsight dashboard to visualize call trends, categories, and flagged events over time.
4.  **Audio Integration:** In production, integrate with actual audio transcription services to populate the transcription table.
5.  **Enhanced Search:** Expand Cortex Search with additional attributes and implement real-time indexing for immediate searchability.
6.  **Knowledge Base:** Use search results to build a knowledge base of common issues and successful resolution patterns.